# Four-Layer Neural Network from Scratch in NumPy

**Author:** Vishwajeet Adkine

This notebook builds a fully-connected neural network using **only NumPy** — no PyTorch, TensorFlow or Keras for the model itself. It implements:

- **Forward propagation** (ReLU hidden layers, softmax output)
- **Backpropagation** (derived by hand, verified with numerical gradient checking)
- **Mini-batch SGD** (shuffling, learning-rate decay, optional L2 regularisation)

**Task:** handwritten digit classification (MNIST; falls back to scikit-learn's 8×8 digits if MNIST can't be downloaded).

**Architecture:** `input → 128 → 64 → 32 → 10` — four weight layers (three hidden + one output).

---
**Contents**
1. Setup
2. Data
3. Network implementation
4. Gradient check
5. Training
6. Evaluation
7. Save / load weights

## 1. Setup

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt

SEED = 42
np.random.seed(SEED)
plt.rcParams["figure.dpi"] = 110
print("NumPy", np.__version__)

## 2. Data

MNIST is loaded through `tensorflow.keras.datasets` (it is pre-installed on Colab, and is used **only** to download the data). Pixels are scaled to `[0, 1]` and flattened. We hold out 10% of the training set for validation.

In [ ]:
def load_data():
    try:
        from tensorflow.keras.datasets import mnist
        (X_train, y_train), (X_test, y_test) = mnist.load_data()
        X_train = X_train.reshape(len(X_train), -1).astype(np.float64) / 255.0
        X_test = X_test.reshape(len(X_test), -1).astype(np.float64) / 255.0
        name, side = "MNIST", 28
    except Exception as e:
        print("MNIST unavailable, falling back to sklearn digits:", type(e).__name__)
        from sklearn.datasets import load_digits
        from sklearn.model_selection import train_test_split
        d = load_digits()
        X = d.data.astype(np.float64) / 16.0
        X_train, X_test, y_train, y_test = train_test_split(
            X, d.target, test_size=0.2, random_state=SEED, stratify=d.target)
        name, side = "sklearn digits (8x8)", 8
    return X_train, y_train.astype(int), X_test, y_test.astype(int), name, side

X_train_full, y_train_full, X_test, y_test, DATASET, SIDE = load_data()

# train / validation split
rng = np.random.default_rng(SEED)
idx = rng.permutation(len(X_train_full))
n_val = int(0.1 * len(idx))
val_idx, tr_idx = idx[:n_val], idx[n_val:]
X_val, y_val = X_train_full[val_idx], y_train_full[val_idx]
X_train, y_train = X_train_full[tr_idx], y_train_full[tr_idx]

print(f"Dataset : {DATASET}")
print(f"Train   : {X_train.shape}   Val: {X_val.shape}   Test: {X_test.shape}")

In [ ]:
fig, axes = plt.subplots(2, 8, figsize=(12, 3.4))
for ax, img, lab in zip(axes.ravel(), X_train, y_train):
    ax.imshow(img.reshape(SIDE, SIDE), cmap="gray")
    ax.set_title(int(lab), fontsize=9)
    ax.axis("off")
plt.suptitle(f"Sample images — {DATASET}")
plt.tight_layout()
plt.show()

## 3. Network implementation

### The maths

For layer $l = 1,\dots,L$ (here $L=4$) with $a^{[0]} = X$:

$$z^{[l]} = a^{[l-1]} W^{[l]} + b^{[l]}, \qquad a^{[l]} = \mathrm{ReLU}(z^{[l]}) \;\; (l<L), \qquad a^{[L]} = \mathrm{softmax}(z^{[L]})$$

**Loss** (cross-entropy over a batch of $m$ examples, plus optional L2):

$$J = -\frac{1}{m}\sum_{i}\sum_{k} y_{ik}\log a^{[L]}_{ik} + \frac{\lambda}{2m}\sum_l \lVert W^{[l]}\rVert_F^2$$

**Backpropagation.** With softmax + cross-entropy the output error simplifies to

$$\delta^{[L]} = \frac{a^{[L]} - y}{m}$$

and for earlier layers

$$\delta^{[l]} = \big(\delta^{[l+1]} (W^{[l+1]})^\top\big) \odot \mathbb{1}[z^{[l]}>0]$$

$$\frac{\partial J}{\partial W^{[l]}} = (a^{[l-1]})^\top \delta^{[l]} + \frac{\lambda}{m}W^{[l]}, \qquad \frac{\partial J}{\partial b^{[l]}} = \sum_i \delta^{[l]}_i$$

**Mini-batch SGD update:** $\;W \leftarrow W - \eta\,\partial J/\partial W$, with $\eta$ decayed each epoch.

**Initialisation:** He initialisation, $W \sim \mathcal{N}(0, 2/n_{in})$, which suits ReLU.

In [ ]:
def one_hot(y, n_classes):
    out = np.zeros((len(y), n_classes))
    out[np.arange(len(y)), y] = 1.0
    return out

def relu(z):
    return np.maximum(0.0, z)

def softmax(z):
    z = z - z.max(axis=1, keepdims=True)      # numerical stability
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)


class NeuralNetwork:
    """Fully-connected network: ReLU hidden layers, softmax output, trained with mini-batch SGD."""

    def __init__(self, layer_sizes, l2=0.0, seed=SEED):
        self.sizes = list(layer_sizes)
        self.l2 = l2
        rng = np.random.default_rng(seed)
        self.W = [rng.standard_normal((i, o)) * np.sqrt(2.0 / i)
                  for i, o in zip(self.sizes[:-1], self.sizes[1:])]
        self.b = [np.zeros((1, o)) for o in self.sizes[1:]]
        self.history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": [], "lr": []}

    # ---------- forward ----------
    def forward(self, X):
        self.a = [X]          # activations (a[0] = input)
        self.z = []           # pre-activations
        for l, (W, b) in enumerate(zip(self.W, self.b)):
            z = self.a[-1] @ W + b
            self.z.append(z)
            self.a.append(softmax(z) if l == len(self.W) - 1 else relu(z))
        return self.a[-1]

    # ---------- loss ----------
    def loss(self, probs, Y):
        m = Y.shape[0]
        ce = -np.sum(Y * np.log(probs + 1e-12)) / m
        reg = 0.5 * self.l2 / m * sum(np.sum(W ** 2) for W in self.W)
        return ce + reg

    # ---------- backward ----------
    def backward(self, Y):
        m = Y.shape[0]
        L = len(self.W)
        dW, db = [None] * L, [None] * L
        delta = (self.a[-1] - Y) / m                         # softmax + cross-entropy
        for l in reversed(range(L)):
            dW[l] = self.a[l].T @ delta + (self.l2 / m) * self.W[l]
            db[l] = delta.sum(axis=0, keepdims=True)
            if l > 0:
                delta = (delta @ self.W[l].T) * (self.z[l - 1] > 0)   # ReLU derivative
        return dW, db

    # ---------- SGD step ----------
    def step(self, dW, db, lr):
        for l in range(len(self.W)):
            self.W[l] -= lr * dW[l]
            self.b[l] -= lr * db[l]

    # ---------- inference ----------
    def predict_proba(self, X, batch_size=2048):
        out = [self.forward(X[i:i + batch_size]) for i in range(0, len(X), batch_size)]
        return np.vstack(out)

    def predict(self, X):
        return self.predict_proba(X).argmax(axis=1)

    def evaluate(self, X, y):
        probs = self.predict_proba(X)
        return self.loss(probs, one_hot(y, self.sizes[-1])), np.mean(probs.argmax(axis=1) == y)

    # ---------- training loop ----------
    def fit(self, X, y, X_val=None, y_val=None, epochs=30, batch_size=128,
            lr=0.1, lr_decay=0.95, seed=SEED, verbose=True):
        n_classes = self.sizes[-1]
        Y = one_hot(y, n_classes)
        rng = np.random.default_rng(seed)
        n = X.shape[0]
        for epoch in range(1, epochs + 1):
            t0 = time.time()
            perm = rng.permutation(n)                         # reshuffle every epoch
            Xs, Ys = X[perm], Y[perm]
            for i in range(0, n, batch_size):                 # mini-batches
                xb, yb = Xs[i:i + batch_size], Ys[i:i + batch_size]
                self.forward(xb)
                dW, db = self.backward(yb)
                self.step(dW, db, lr)

            tr_loss, tr_acc = self.evaluate(X, y)
            self.history["train_loss"].append(tr_loss)
            self.history["train_acc"].append(tr_acc)
            self.history["lr"].append(lr)
            msg = f"Epoch {epoch:3d}/{epochs} | lr {lr:.4f} | loss {tr_loss:.4f} | acc {tr_acc:.4f}"
            if X_val is not None:
                v_loss, v_acc = self.evaluate(X_val, y_val)
                self.history["val_loss"].append(v_loss)
                self.history["val_acc"].append(v_acc)
                msg += f" | val_loss {v_loss:.4f} | val_acc {v_acc:.4f}"
            if verbose and (epoch == 1 or epoch % 5 == 0 or epoch == epochs):
                print(msg + f" | {time.time() - t0:.2f}s")
            lr *= lr_decay
        return self.history

## 4. Gradient check

Before training, we verify backprop against **numerical gradients** (centred finite differences):

$$\frac{\partial J}{\partial \theta} \approx \frac{J(\theta+\varepsilon) - J(\theta-\varepsilon)}{2\varepsilon}$$

A relative error around $10^{-7}$ or smaller means the analytic gradients are correct.

In [ ]:
def gradient_check(net, X, y, eps=1e-5, n_checks=5, seed=0):
    Y = one_hot(y, net.sizes[-1])
    net.forward(X)
    dW, db = net.backward(Y)
    rng = np.random.default_rng(seed)
    worst = 0.0
    for l in range(len(net.W)):
        for _ in range(n_checks):
            i = rng.integers(net.W[l].shape[0]); j = rng.integers(net.W[l].shape[1])
            orig = net.W[l][i, j]
            net.W[l][i, j] = orig + eps; Jp = net.loss(net.forward(X), Y)
            net.W[l][i, j] = orig - eps; Jm = net.loss(net.forward(X), Y)
            net.W[l][i, j] = orig
            num = (Jp - Jm) / (2 * eps)
            ana = dW[l][i, j]
            rel = abs(num - ana) / max(abs(num) + abs(ana), 1e-12)
            worst = max(worst, rel)
    return worst

check_net = NeuralNetwork([X_train.shape[1], 16, 12, 8, 10], l2=0.1)   # small net, float64
err = gradient_check(check_net, X_train[:20], y_train[:20])
print(f"Max relative error: {err:.2e}")
print("Gradient check PASSED ✅" if err < 1e-6 else "Gradient check FAILED ❌")

## 5. Training

Architecture: `input → 128 → 64 → 32 → 10`.
Hyper-parameters are collected in one place so they are easy to tweak.

In [ ]:
INPUT_DIM = X_train.shape[1]
N_CLASSES = 10

CONFIG = dict(
    layer_sizes=[INPUT_DIM, 128, 64, 32, N_CLASSES],
    epochs=30,
    batch_size=128,
    lr=0.1,
    lr_decay=0.95,
    l2=1e-4,
)

net = NeuralNetwork(CONFIG["layer_sizes"], l2=CONFIG["l2"])
n_params = sum(W.size + b.size for W, b in zip(net.W, net.b))
print("Layers:", CONFIG["layer_sizes"], f"| parameters: {n_params:,}")

start = time.time()
history = net.fit(X_train, y_train, X_val, y_val,
                  epochs=CONFIG["epochs"], batch_size=CONFIG["batch_size"],
                  lr=CONFIG["lr"], lr_decay=CONFIG["lr_decay"])
print(f"\nTraining time: {time.time() - start:.1f}s")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ep = np.arange(1, len(history["train_loss"]) + 1)

ax[0].plot(ep, history["train_loss"], label="train")
ax[0].plot(ep, history["val_loss"], label="validation")
ax[0].set(title="Loss", xlabel="Epoch", ylabel="Cross-entropy"); ax[0].legend(); ax[0].grid(alpha=.3)

ax[1].plot(ep, history["train_acc"], label="train")
ax[1].plot(ep, history["val_acc"], label="validation")
ax[1].set(title="Accuracy", xlabel="Epoch", ylabel="Accuracy"); ax[1].legend(); ax[1].grid(alpha=.3)

plt.tight_layout(); plt.show()

## 6. Evaluation

In [ ]:
test_loss, test_acc = net.evaluate(X_test, y_test)
print(f"Test loss    : {test_loss:.4f}")
print(f"Test accuracy: {test_acc * 100:.2f}%")

In [ ]:
def confusion_matrix(y_true, y_pred, n=10):
    cm = np.zeros((n, n), dtype=int)
    np.add.at(cm, (y_true, y_pred), 1)
    return cm

y_pred = net.predict(X_test)
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(6.5, 5.5))
im = ax.imshow(cm, cmap="Blues")
ax.set(xticks=range(10), yticks=range(10), xlabel="Predicted", ylabel="True", title="Confusion matrix (test set)")
for i in range(10):
    for j in range(10):
        ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=7,
                color="white" if cm[i, j] > cm.max() / 2 else "black")
plt.colorbar(im); plt.tight_layout(); plt.show()

# per-class precision / recall / F1
print(f"{'class':>5} {'precision':>10} {'recall':>8} {'f1':>8}")
for c in range(10):
    tp = cm[c, c]; fp = cm[:, c].sum() - tp; fn = cm[c, :].sum() - tp
    p = tp / max(tp + fp, 1); r = tp / max(tp + fn, 1); f = 2 * p * r / max(p + r, 1e-12)
    print(f"{c:>5} {p:>10.3f} {r:>8.3f} {f:>8.3f}")

In [ ]:
wrong = np.where(y_pred != y_test)[0]
probs = net.predict_proba(X_test[wrong[:16]])
print(f"{len(wrong)} misclassified out of {len(y_test)}")

fig, axes = plt.subplots(2, 8, figsize=(12, 3.6))
for ax, k, p in zip(axes.ravel(), wrong[:16], probs):
    ax.imshow(X_test[k].reshape(SIDE, SIDE), cmap="gray")
    ax.set_title(f"true {y_test[k]}\npred {p.argmax()} ({p.max():.2f})", fontsize=8)
    ax.axis("off")
plt.suptitle("Misclassified test examples"); plt.tight_layout(); plt.show()

### Optional experiment — effect of batch size

Smaller batches give noisier but more frequent updates; larger batches are smoother but make fewer updates per epoch. Run this cell to compare (about a minute on Colab).

In [ ]:
results = {}
for bs in [16, 64, 256, 1024]:
    m = NeuralNetwork(CONFIG["layer_sizes"], l2=CONFIG["l2"])
    h = m.fit(X_train, y_train, X_val, y_val, epochs=10, batch_size=bs,
              lr=CONFIG["lr"], lr_decay=CONFIG["lr_decay"], verbose=False)
    results[bs] = h["val_acc"]

plt.figure(figsize=(7, 4))
for bs, accs in results.items():
    plt.plot(range(1, len(accs) + 1), accs, label=f"batch {bs}")
plt.xlabel("Epoch"); plt.ylabel("Validation accuracy"); plt.title("Batch-size comparison")
plt.legend(); plt.grid(alpha=.3); plt.show()

## 7. Save / load weights

In [ ]:
def save_weights(net, path="nn_weights.npz"):
    arrays = {f"W{i}": W for i, W in enumerate(net.W)}
    arrays.update({f"b{i}": b for i, b in enumerate(net.b)})
    arrays["sizes"] = np.array(net.sizes)
    np.savez(path, **arrays)

def load_weights(path="nn_weights.npz"):
    d = np.load(path)
    n = NeuralNetwork(d["sizes"].tolist())
    n.W = [d[f"W{i}"] for i in range(len(n.sizes) - 1)]
    n.b = [d[f"b{i}"] for i in range(len(n.sizes) - 1)]
    return n

save_weights(net)
restored = load_weights()
assert np.allclose(restored.predict_proba(X_test[:100]), net.predict_proba(X_test[:100]))
print("Weights saved to nn_weights.npz and reloaded successfully ✅")

# In Colab you can download the file with:
# from google.colab import files; files.download("nn_weights.npz")

## Summary

- Implemented a four-layer MLP, forward pass, backprop and mini-batch SGD **entirely in NumPy**.
- Verified gradients numerically before training.
- Tracked loss/accuracy per epoch, evaluated with a confusion matrix and per-class metrics.

**Ideas to extend:** momentum / Adam, dropout, batch normalisation, learning-rate schedules, other datasets (Fashion-MNIST).